# Practice
Model I/O부터 Memory까지 연습하고 넘어가 봅시다🐿️

In [1]:
# 환경 설정 (open api key 사용하기 위함)
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

### 1. Chain을 이용한 Simple LLM

1. PromptTemplate - System message, User message
2. LLM (model)
3. OutputParser
- Chain ~> 간단한 질의를 보내 응답 텍스트만 출력

In [3]:
# 1. PromptTemplate 생성
from langchain import PromptTemplate

system_message = "너는 위트있게 단답하는 챗봇이야. 이럴 때는 어떻게 해야할까?\n{emotion}"

prompt_tpl = PromptTemplate(
    template = system_message,
    input_variables = ['emotion'],
)

user_message = '내 기분이 너무 안 좋아.'
prompt = prompt_tpl.format(emotion=user_message)

In [9]:
# 2. LLM 모델 요청을 위한 인스턴스 생성
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

endpoint = HuggingFaceEndpoint(
    repo_id='MLP-KTLim/llama-3-Korean-Bllossom-8B', # conversational 태그가 있는 모델만 가능함
    taks='text-generation',
    max_new_tokens=1024,
    huggingfacehub_api_token=HF_TOKEN
)

hf_model = ChatHuggingFace(
    llm=endpoint,
    verbos=True
)

response = hf_model.invoke(prompt)
print(response.content)

WARNING! taks is not default parameter.
                    taks was transferred to model_kwargs.
                    Please make sure that taks is what you intended.


기분이 안 좋으신가요? 그럼 기분을 풀어줄 만한 재미있는 동영상이나 GIF를 보내드릴까요?


In [10]:
# 3. 응답 문자열만 출력학 위한 OutputParser 생성
from langchain_core.output_parsers import StrOutputParser

string_parser = StrOutputParser()
output = string_parser.parse(response.content)
print(output)

기분이 안 좋으신가요? 그럼 기분을 풀어줄 만한 재미있는 동영상이나 GIF를 보내드릴까요?


### 2. 단계별 Chatbot
- 내 이름을 알려주고, 내 이름이 뭐냐고 물어보기

1. 그냥 Chat ::: ChatOpenAI, HumanMessage

2. 직접 대화 맥락 유지 ::: ChatOpenAI, HumanMessage, AIMessage

3. Memory로 대화 맥락 유지
- langchain_openai의 ChatOpenAI
- langchain_core.message의 클래스
- langchain_core.runnables의 클래스
- langchain_core.prompts의 클래스